# Bài 01 — Vector, ma trận và tư duy về shape với NumPy

## Mục tiêu

Sau notebook này, bạn có thể:

- tạo và kiểm tra scalar, vector, matrix, tensor;
- đọc `shape`, `ndim`, `size`, `dtype` và giải thích ý nghĩa từng trục;
- index, slice, reshape, transpose và chọn đúng `axis`;
- phân biệt `(n,)`, `(1, n)`, `(n, 1)`;
- dùng broadcasting để chuẩn hóa dữ liệu;
- tránh lỗi do view/copy;
- hoàn thành một bài tổng hợp tiền xử lý dữ liệu kiểu ML.

> **Cách học:** trước mỗi ô code, hãy ghi dự đoán shape và output ra giấy. Chỉ sau đó mới chạy ô.

## 0. Khởi tạo

Notebook chỉ cần NumPy. Seed `42` giúp kết quả ngẫu nhiên có thể tái lập. Hàm `describe` in bốn thuộc tính quan trọng của một mảng.

In [1]:
import numpy as np

rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)

def describe(name, array):
    array = np.asarray(array)
    print(
        f"{name:>10} | shape={str(array.shape):<10} "
        f"ndim={array.ndim} size={array.size:<3} dtype={array.dtype}"
    )

print("NumPy version:", np.__version__)

NumPy version: 2.4.3


## 1. Scalar → vector → matrix → tensor

Số phần tử và số trục là hai khái niệm khác nhau. Vector 100 phần tử vẫn chỉ có một trục. Hãy dự đoán `shape`, `ndim`, `size` của bốn biến dưới đây.

In [2]:
scalar = np.array(3.14)
vector = np.array([2.0, 5.0, -1.0])
matrix = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
tensor = np.zeros((2, 4, 4, 3))  # (batch, height, width, channels)

for name, value in [("scalar", scalar), ("vector", vector),
                    ("matrix", matrix), ("tensor", tensor)]:
    describe(name, value)

assert scalar.shape == ()
assert vector.shape == (3,)
assert matrix.shape == (2, 3)
assert tensor.ndim == 4

    scalar | shape=()         ndim=0 size=1   dtype=float64
    vector | shape=(3,)       ndim=1 size=3   dtype=float64
    matrix | shape=(2, 3)     ndim=2 size=6   dtype=float64
    tensor | shape=(2, 4, 4, 3) ndim=4 size=96  dtype=float64


### Đọc shape bằng ý nghĩa, không chỉ bằng con số

`(2, 4, 4, 3)` chỉ hữu ích khi ta biết các trục lần lượt là `(batch, height, width, channels)`. Trong ML, nên ghi chú ý nghĩa trục tại nơi tạo dữ liệu.

## 2. Tạo mảng và kiểm soát `dtype`

NumPy có nhiều cách tạo mảng. `arange` phù hợp với dãy cách đều kiểu số nguyên; `linspace` phù hợp khi cần một số lượng điểm xác định trên một đoạn.

In [3]:
integers = np.arange(0, 10, 2)
grid = np.linspace(0.0, 1.0, num=5)
zeros = np.zeros((2, 3), dtype=float)
ones = np.ones((2, 3), dtype=np.int64)
random_matrix = rng.normal(loc=0.0, scale=1.0, size=(3, 2))

for name, value in [("integers", integers), ("grid", grid),
                    ("zeros", zeros), ("ones", ones),
                    ("random", random_matrix)]:
    describe(name, value)

features = integers.astype(float)
assert np.issubdtype(features.dtype, np.floating)

  integers | shape=(5,)       ndim=1 size=5   dtype=int64
      grid | shape=(5,)       ndim=1 size=5   dtype=float64
     zeros | shape=(2, 3)     ndim=2 size=6   dtype=float64
      ones | shape=(2, 3)     ndim=2 size=6   dtype=int64
    random | shape=(3, 2)     ndim=2 size=6   dtype=float64


## 3. Dataset dạng bảng: hàng là mẫu, cột là đặc trưng

Dataset dưới đây có ba cột: `hours_studied`, `hours_slept`, `previous_score`. Hãy đọc từng lệnh slicing thành lời trước khi chạy.

In [4]:
X = np.array([
    [2.0, 7.0, 60.0],
    [4.0, 6.0, 72.0],
    [6.0, 8.0, 85.0],
    [8.0, 7.0, 92.0],
])
feature_names = np.array(["hours_studied", "hours_slept", "previous_score"])

first_sample = X[0]
last_sample = X[-1]
sleep_feature_1d = X[:, 1]
sleep_feature_2d = X[:, 1:2]
middle_block = X[1:3, :2]
high_previous_score = X[X[:, 2] >= 80]

for name, value in [("X", X), ("first", first_sample),
                    ("sleep 1d", sleep_feature_1d),
                    ("sleep 2d", sleep_feature_2d),
                    ("block", middle_block), ("filtered", high_previous_score)]:
    describe(name, value)

         X | shape=(4, 3)     ndim=2 size=12  dtype=float64
     first | shape=(3,)       ndim=1 size=3   dtype=float64
  sleep 1d | shape=(4,)       ndim=1 size=4   dtype=float64
  sleep 2d | shape=(4, 1)     ndim=2 size=4   dtype=float64
     block | shape=(2, 2)     ndim=2 size=4   dtype=float64
  filtered | shape=(2, 3)     ndim=2 size=6   dtype=float64


### Index nguyên làm mất trục; slice giữ trục

`X[:, 1]` và `X[:, 1:2]` chứa cùng bốn giá trị nhưng shape khác nhau. Đây là một trong những nguồn lỗi phổ biến nhất khi ghép các bước xử lý.

In [5]:
print("1D:", sleep_feature_1d, sleep_feature_1d.shape)
print("2D:\n", sleep_feature_2d, sleep_feature_2d.shape)

assert sleep_feature_1d.shape == (4,)
assert sleep_feature_2d.shape == (4, 1)
np.testing.assert_array_equal(sleep_feature_1d, sleep_feature_2d[:, 0])

1D: [7. 6. 8. 7.] (4,)
2D:
 [[7.]
 [6.]
 [8.]
 [7.]] (4, 1)


## 4. `(n,)`, `(1, n)` và `(n, 1)`

Một vector NumPy 1D chưa có hướng hàng/cột. Do chỉ có một trục, `.T` không làm thay đổi nó. Muốn có hàng hoặc cột, phải thêm một trục.

In [6]:
x = np.array([10.0, 20.0, 30.0])
row = x.reshape(1, -1)       # tương đương x[None, :]
column = x.reshape(-1, 1)    # tương đương x[:, None]

for name, value in [("x", x), ("x.T", x.T),
                    ("row", row), ("column", column)]:
    describe(name, value)

assert x.T.shape == x.shape
assert row.shape == (1, 3)
assert column.shape == (3, 1)

         x | shape=(3,)       ndim=1 size=3   dtype=float64
       x.T | shape=(3,)       ndim=1 size=3   dtype=float64
       row | shape=(1, 3)     ndim=2 size=3   dtype=float64
    column | shape=(3, 1)     ndim=2 size=3   dtype=float64


## 5. `reshape` và chuyển vị

`reshape` thay đổi cách diễn giải cùng số phần tử. Ký hiệu `-1` cho phép NumPy suy ra đúng một kích thước. Chuyển vị ma trận hoán đổi hàng và cột.

In [7]:
sequence = np.arange(12)
A = sequence.reshape(3, -1)
A_T = A.T
flattened = A.reshape(-1)

print("A =\n", A)
print("A.T =\n", A_T)
describe("sequence", sequence)
describe("A", A)
describe("A.T", A_T)
describe("flattened", flattened)

assert A.shape == (3, 4)
assert A_T.shape == (4, 3)
assert A.size == sequence.size
np.testing.assert_array_equal(flattened, sequence)

A =
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
A.T =
 [[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]
  sequence | shape=(12,)      ndim=1 size=12  dtype=int64
         A | shape=(3, 4)     ndim=2 size=12  dtype=int64
       A.T | shape=(4, 3)     ndim=2 size=12  dtype=int64
 flattened | shape=(12,)      ndim=1 size=12  dtype=int64


### Hoán đổi trục của tensor ảnh

Các thư viện có thể dùng `(batch, height, width, channels)` hoặc `(batch, channels, height, width)`. `transpose` cho tensor nhiều chiều cần nêu rõ thứ tự trục mới.

In [8]:
images_nhwc = np.zeros((8, 28, 28, 3))
images_nchw = images_nhwc.transpose(0, 3, 1, 2)

describe("NHWC", images_nhwc)
describe("NCHW", images_nchw)
assert images_nchw.shape == (8, 3, 28, 28)

      NHWC | shape=(8, 28, 28, 3) ndim=4 size=18816 dtype=float64
      NCHW | shape=(8, 3, 28, 28) ndim=4 size=18816 dtype=float64


## 6. Hiểu `axis` bằng câu hỏi “trục nào biến mất?”

Với `X.shape == (samples, features)`, `axis=0` gộp trục samples và để lại một kết quả cho mỗi feature. `axis=1` gộp trục features và để lại một kết quả cho mỗi sample.

In [9]:
feature_means = X.mean(axis=0)
sample_means = X.mean(axis=1)
feature_means_keepdims = X.mean(axis=0, keepdims=True)

print("feature means:", feature_means)
print("sample means: ", sample_means)
describe("axis=0", feature_means)
describe("axis=1", sample_means)
describe("keepdims", feature_means_keepdims)

assert feature_means.shape == (3,)
assert sample_means.shape == (4,)
assert feature_means_keepdims.shape == (1, 3)

feature means: [ 5.    7.   77.25]
sample means:  [23.    27.333 33.    35.667]
    axis=0 | shape=(3,)       ndim=1 size=3   dtype=float64
    axis=1 | shape=(4,)       ndim=1 size=4   dtype=float64
  keepdims | shape=(1, 3)     ndim=2 size=3   dtype=float64


## 7. Broadcasting qua bài toán chuẩn hóa

Ta chuẩn hóa từng cột theo công thức `Z = (X - mean) / std`. Hai vector `(3,)` được broadcast qua bốn hàng của ma trận `(4, 3)`.

In [10]:
mu = X.mean(axis=0)           # (features,)
sigma = X.std(axis=0)         # (features,)
Z = (X - mu) / sigma          # (samples, features)

print("mu:   ", mu)
print("sigma:", sigma)
print("Z =\n", Z)
print("mean after standardization:", Z.mean(axis=0))
print("std after standardization: ", Z.std(axis=0))

assert Z.shape == X.shape
np.testing.assert_allclose(Z.mean(axis=0), 0.0, atol=1e-12)
np.testing.assert_allclose(Z.std(axis=0), 1.0, atol=1e-12)

mu:    [ 5.    7.   77.25]
sigma: [ 2.236  0.707 12.275]
Z =
 [[-1.342  0.    -1.405]
 [-0.447 -1.414 -0.428]
 [ 0.447  1.414  0.631]
 [ 1.342  0.     1.202]]
mean after standardization: [0. 0. 0.]
std after standardization:  [1. 1. 1.]


### Bẫy broadcasting

`X - X.mean(axis=1)` không chạy với `X.shape == (4, 3)` vì `(4, 3)` và `(4,)` không tương thích khi so từ phải sang trái. Nếu thật sự muốn trừ trung bình từng hàng, cần giữ shape `(4, 1)` bằng `keepdims=True`.

In [11]:
row_means = X.mean(axis=1, keepdims=True)  # (4, 1)
row_centered = X - row_means               # broadcast (4, 1) -> (4, 3)

describe("row means", row_means)
describe("centered", row_centered)
np.testing.assert_allclose(row_centered.mean(axis=1), 0.0, atol=1e-12)

 row means | shape=(4, 1)     ndim=2 size=4   dtype=float64
  centered | shape=(4, 3)     ndim=2 size=12  dtype=float64


## 8. View và copy

Slicing cơ bản thường tạo view dùng chung vùng nhớ. Hãy quan sát mảng gốc thay đổi khi sửa view, rồi so sánh với `.copy()`.

In [12]:
original = np.arange(12).reshape(4, 3)
view = original[:2]
view[0, 0] = -999
print("Sau khi sửa view, original[0, 0] =", original[0, 0])
assert original[0, 0] == -999

independent = original[:2].copy()
independent[0, 0] = 12345
print("Sau khi sửa copy, original[0, 0] =", original[0, 0])
assert original[0, 0] == -999

Sau khi sửa view, original[0, 0] = -999
Sau khi sửa copy, original[0, 0] = -999


## Bài tập

Có 6 bài tập tăng dần. Mỗi ô bài tập chạy được ngay cả khi chưa làm; ô kiểm tra sẽ nhắc biến còn thiếu thay vì làm notebook dừng. Hãy:

1. viết dự đoán shape;
2. bỏ dấu `#` và hoàn thành dòng `TODO`;
3. chạy ô bài tập rồi ô kiểm tra;
4. chỉ mở đáp án sau khi đã thử.

### Bài 1 — Tạo dataset và đọc thuộc tính

Tạo `ex1_data` gồm các số từ 0 đến 23, shape `(6, 4)`, dtype float. Sau đó tạo `ex1_last_sample` là hàng cuối, shape `(4,)`.

In [13]:
# TODO: bỏ dấu # và điền biểu thức
# ex1_data = ...
# ex1_last_sample = ...

In [14]:
if "ex1_data" not in globals() or "ex1_last_sample" not in globals():
    print("⏳ Chưa làm Bài 1: hãy tạo ex1_data và ex1_last_sample.")
else:
    assert ex1_data.shape == (6, 4)
    assert np.issubdtype(ex1_data.dtype, np.floating)
    assert ex1_last_sample.shape == (4,)
    np.testing.assert_array_equal(ex1_last_sample, [20, 21, 22, 23])
    print("✅ Bài 1 đúng")

⏳ Chưa làm Bài 1: hãy tạo ex1_data và ex1_last_sample.


<details><summary>Đáp án Bài 1</summary>

```python
ex1_data = np.arange(24, dtype=float).reshape(6, 4)
ex1_last_sample = ex1_data[-1]
```
</details>

### Bài 2 — Index, slice và boolean mask

Từ `exercise_X` bên dưới:

- lấy cột thứ ba thành `ex2_col` có shape `(5, 1)`;
- lấy các hàng có giá trị cột đầu lớn hơn 5 thành `ex2_filtered`.

In [15]:
exercise_X = np.arange(20).reshape(5, 4)
print(exercise_X)

# TODO1
# ex2_col = ...
# ex2_filtered = ...


[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]
 [16 17 18 19]]


In [16]:
if "ex2_col" not in globals() or "ex2_filtered" not in globals():
    print("⏳ Chưa làm Bài 2: hãy tạo ex2_col và ex2_filtered.")
else:
    assert ex2_col.shape == (5, 1)
    assert ex2_filtered.shape == (3, 4)
    np.testing.assert_array_equal(ex2_col[:, 0], [2, 6, 10, 14, 18])
    np.testing.assert_array_equal(ex2_filtered[:, 0], [8, 12, 16])
    print("✅ Bài 2 đúng")

⏳ Chưa làm Bài 2: hãy tạo ex2_col và ex2_filtered.


<details><summary>Đáp án Bài 2</summary>

```python
ex2_col = exercise_X[:, 2:3]
ex2_filtered = exercise_X[exercise_X[:, 0] > 5]
```
</details>

### Bài 3 — Đổi quy ước trục ảnh

`batch_images` có shape `(10, 32, 32, 3)` theo NHWC. Tạo `ex3_nchw` có shape `(10, 3, 32, 32)` mà không thay đổi dữ liệu.

In [17]:
batch_images = rng.integers(0, 256, size=(10, 32, 32, 3), dtype=np.uint8)

# TODO
# ex3_nchw = ...

In [18]:
if "ex3_nchw" not in globals():
    print("⏳ Chưa làm Bài 3: hãy tạo ex3_nchw.")
else:
    assert ex3_nchw.shape == (10, 3, 32, 32)
    np.testing.assert_array_equal(ex3_nchw[0, :, 0, 0], batch_images[0, 0, 0, :])
    print("✅ Bài 3 đúng")

⏳ Chưa làm Bài 3: hãy tạo ex3_nchw.


<details><summary>Đáp án Bài 3</summary>

```python
ex3_nchw = batch_images.transpose(0, 3, 1, 2)
```
</details>

### Bài 4 — Chuẩn hóa đúng trục

Chuẩn hóa **từng đặc trưng** của `exercise_features`. Tạo `ex4_standardized` sao cho trung bình từng cột bằng 0 và độ lệch chuẩn từng cột bằng 1. Không viết vòng lặp.

In [19]:
exercise_features = rng.normal(loc=[10.0, 100.0, -5.0], scale=[2.0, 20.0, 0.5], size=(100, 3))

# TODO
# ex4_mu = ...
# ex4_sigma = ...
# ex4_standardized = ...

In [20]:
required_ex4 = ["ex4_mu", "ex4_sigma", "ex4_standardized"]
if not all(name in globals() for name in required_ex4):
    print("⏳ Chưa làm Bài 4: hãy tạo ex4_mu, ex4_sigma và ex4_standardized.")
else:
    assert ex4_standardized.shape == exercise_features.shape
    np.testing.assert_allclose(ex4_standardized.mean(axis=0), 0.0, atol=1e-12)
    np.testing.assert_allclose(ex4_standardized.std(axis=0), 1.0, atol=1e-12)
    print("✅ Bài 4 đúng")

⏳ Chưa làm Bài 4: hãy tạo ex4_mu, ex4_sigma và ex4_standardized.


<details><summary>Đáp án Bài 4</summary>

```python
ex4_mu = exercise_features.mean(axis=0)
ex4_sigma = exercise_features.std(axis=0)
ex4_standardized = (exercise_features - ex4_mu) / ex4_sigma
```
</details>

### Bài 5 — Sửa lỗi broadcasting

Mục tiêu là trừ trung bình **của chính mỗi hàng** khỏi hàng đó. Dòng được chú thích sẽ lỗi. Tạo `ex5_centered` đúng shape `(6, 4)` và không dùng vòng lặp.

In [21]:
exercise_rows = np.arange(24, dtype=float).reshape(6, 4)
# broken = exercise_rows - exercise_rows.mean(axis=1)

# TODO
# ex5_centered = ...

In [22]:
if "ex5_centered" not in globals():
    print("⏳ Chưa làm Bài 5: hãy tạo ex5_centered.")
else:
    assert ex5_centered.shape == (6, 4)
    np.testing.assert_allclose(ex5_centered.mean(axis=1), 0.0, atol=1e-12)
    print("✅ Bài 5 đúng")

⏳ Chưa làm Bài 5: hãy tạo ex5_centered.


<details><summary>Đáp án Bài 5</summary>

```python
ex5_row_means = exercise_rows.mean(axis=1, keepdims=True)
ex5_centered = exercise_rows - ex5_row_means
```

Shape `(6, 1)` broadcast được với `(6, 4)`; shape `(6,)` thì không.
</details>

### Bài 6 — Tránh sửa dữ liệu gốc

Tạo `ex6_subset` từ ba hàng đầu của `source_data`. Khi gán `ex6_subset[0, 0] = -1`, `source_data` phải giữ nguyên.

In [23]:
source_data = np.arange(20).reshape(5, 4)
source_before = source_data.copy()

# TODO: tạo bản độc lập rồi bỏ dấu # ở dòng thay đổi
# ex6_subset = ...
# ex6_subset[0, 0] = -1

In [24]:
if "ex6_subset" not in globals():
    print("⏳ Chưa làm Bài 6: hãy tạo ex6_subset.")
else:
    assert ex6_subset.shape == (3, 4)
    assert ex6_subset[0, 0] == -1
    np.testing.assert_array_equal(source_data, source_before)
    print("✅ Bài 6 đúng")

⏳ Chưa làm Bài 6: hãy tạo ex6_subset.


<details><summary>Đáp án Bài 6</summary>

```python
ex6_subset = source_data[:3].copy()
ex6_subset[0, 0] = -1
```
</details>

## Bài tổng hợp — Tiền xử lý dữ liệu cảm biến

Mỗi hàng là một thời điểm; bốn cột là `[temperature, humidity, light, occupancy]`. Hãy:

1. tách 6 hàng đầu thành `cap_train`, 2 hàng cuối thành `cap_test`, dùng copy;
2. tách ba cột feature và cột nhãn occupancy;
3. tính mean/std **chỉ trên train features**;
4. dùng cùng thống kê train để chuẩn hóa cả train và test;
5. lấy các hàng train có người thành `cap_occupied_rows`.

> Đây là nguyên tắc ML quan trọng: không dùng dữ liệu test để tính thống kê tiền xử lý.

In [25]:
sensor_data = np.array([
    [22.0, 45.0, 120.0, 0.0],
    [22.5, 47.0, 300.0, 1.0],
    [23.0, 50.0, 450.0, 1.0],
    [21.5, 44.0, 100.0, 0.0],
    [24.0, 52.0, 500.0, 1.0],
    [23.5, 49.0, 350.0, 1.0],
    [21.0, 43.0, 80.0, 0.0],
    [24.5, 54.0, 550.0, 1.0],
])

# TODO
# cap_train = ...
# cap_test = ...
# cap_X_train, cap_y_train = ..., ...
# cap_X_test, cap_y_test = ..., ...
# cap_mu = ...
# cap_sigma = ...
# cap_Z_train = ...
# cap_Z_test = ...
# cap_occupied_rows = ...

In [26]:
capstone_names = [
    "cap_train", "cap_test", "cap_X_train", "cap_y_train",
    "cap_X_test", "cap_y_test", "cap_mu", "cap_sigma",
    "cap_Z_train", "cap_Z_test", "cap_occupied_rows",
]
if not all(name in globals() for name in capstone_names):
    missing = [name for name in capstone_names if name not in globals()]
    print("⏳ Bài tổng hợp chưa hoàn thành. Còn thiếu:", missing)
else:
    assert cap_train.shape == (6, 4) and cap_test.shape == (2, 4)
    assert cap_X_train.shape == (6, 3) and cap_y_train.shape == (6,)
    assert cap_X_test.shape == (2, 3) and cap_y_test.shape == (2,)
    assert cap_mu.shape == (3,) and cap_sigma.shape == (3,)
    assert cap_Z_train.shape == (6, 3) and cap_Z_test.shape == (2, 3)
    assert cap_occupied_rows.shape == (4, 4)
    np.testing.assert_allclose(cap_Z_train.mean(axis=0), 0.0, atol=1e-12)
    np.testing.assert_allclose(cap_Z_train.std(axis=0), 1.0, atol=1e-12)
    np.testing.assert_array_equal(cap_occupied_rows[:, 3], 1.0)
    print("✅ Hoàn thành bài tổng hợp!")

⏳ Bài tổng hợp chưa hoàn thành. Còn thiếu: ['cap_train', 'cap_test', 'cap_X_train', 'cap_y_train', 'cap_X_test', 'cap_y_test', 'cap_mu', 'cap_sigma', 'cap_Z_train', 'cap_Z_test', 'cap_occupied_rows']


<details><summary>Đáp án bài tổng hợp</summary>

```python
cap_train = sensor_data[:6].copy()
cap_test = sensor_data[6:].copy()

cap_X_train, cap_y_train = cap_train[:, :3], cap_train[:, 3]
cap_X_test, cap_y_test = cap_test[:, :3], cap_test[:, 3]

cap_mu = cap_X_train.mean(axis=0)
cap_sigma = cap_X_train.std(axis=0)
cap_Z_train = (cap_X_train - cap_mu) / cap_sigma
cap_Z_test = (cap_X_test - cap_mu) / cap_sigma
cap_occupied_rows = cap_train[cap_y_train == 1]
```

Lưu ý `cap_Z_test` thường không có mean 0 và std 1, vì nó được biến đổi bằng thống kê của train. Đây là hành vi đúng.
</details>

## Tự kiểm tra nhanh

Không chạy code, hãy dự đoán:

1. `np.zeros((5, 2, 3)).ndim` bằng bao nhiêu?
2. Nếu `A.shape == (5, 3)`, shape của `A[:, -1]` và `A[:, -1:]` là gì?
3. Shape của `A.mean(axis=0, keepdims=True)` là gì?
4. Vì sao `np.ones((5, 3)) + np.ones((5,))` lỗi broadcasting?
5. Khi nào cần `.copy()`?

<details><summary>Đáp án</summary>

1. `3`.
2. `(5,)` và `(5, 1)`.
3. `(1, 3)`.
4. So từ phải sang trái, kích thước `3` và `5` không bằng nhau và không cái nào bằng `1`.
5. Khi cần sửa mảng con mà không muốn làm thay đổi mảng gốc.
</details>

## Tổng kết

- Shape không chỉ là kích thước; mỗi trục phải có ý nghĩa.
- Index nguyên thường làm mất trục, slice giữ trục.
- Vector `(n,)` không tự động là hàng hay cột; `.T` không đổi shape của nó.
- Với phép gộp, trục ghi trong `axis` sẽ biến mất nếu không dùng `keepdims=True`.
- Broadcasting so shape từ phải sang trái.
- Slicing thường tạo view; dùng `.copy()` khi cần dữ liệu độc lập.

Trước khi sang Bài 02, hãy tự viết lại sự khác nhau giữa `(n,)`, `(1, n)`, `(n, 1)` và mô tả một lỗi shape bạn đã tránh được trong notebook này.